In [1]:
# === FastAPI Core ===
from fastapi import FastAPI, Request, UploadFile, File, HTTPException, Depends
from fastapi.responses import StreamingResponse, FileResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles

# === Security & Auth ===
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, JWTError
import bcrypt

# === Pydantic Models ===
from pydantic import BaseModel, EmailStr

# === Database & Files ===
import pymysql
import pymysql.cursors
from openpyxl import load_workbook

# === LLMs & Vector Store ===
import ollama
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.faiss import FaissVectorStore

# === Utilities ===
from datetime import datetime, timedelta
from urllib.parse import quote
from enum import Enum
from typing import Generator
import re
import faiss
import io
import os
import shutil 


c:\Users\txcjs\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
PROJECT_ROOT = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\gpu'

Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2")
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2")
remote_base_url = "http://localhost:11434"
# Instantiate the Ollama LLM
llm = Ollama(model="llama3.2:1b", request_timeout=120.0, temperature=0, context_window=4096, base_url=remote_base_url)
Settings.llm = llm

# FAISS index loading
persist_dir = os.path.join(PROJECT_ROOT, "pipeline", "data", "Embedded")
faiss_path = "faiss.index"
faiss_index_path = os.path.join(persist_dir, faiss_path)
if os.path.exists(faiss_index_path):
    faiss_index = faiss.read_index(faiss_index_path)
    vector_store = FaissVectorStore(faiss_index=faiss_index)
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir, vector_store=vector_store)
    index = load_index_from_storage(storage_context)
    query_engine = index.as_query_engine(similarity_top_k=5, streaming=True)
else:
    faiss_index = None
    vector_store = None
    storage_context = None
    index = None
    query_engine = None
    print(f"WARNING: FAISS index not found at {faiss_index_path}. Vector search will be unavailable until the index is built.")

# This is llama-mini

In [7]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=1500)

chat_engine = index.as_chat_engine(
    chat_mode="context",
    memory=memory,
    system_prompt=(
        "You are a chatbot, able to have normal interactions, as well as talk"
        " about sunshine and rainbows."
    ),
)



In [8]:
response = chat_engine.chat("Hello! It is 3am in Singapore")
print(response.response)

What a lovely time to be awake! The sun might not be shining brightly over Singapore at this hour, but I'm sure the city-state's vibrant atmosphere is still buzzing with energy. How about we talk about something bright and cheerful instead? Did you know that Singapore has some of the most beautiful gardens in the world, like the Gardens by the Bay? Or perhaps you'd rather chat about rainbows - after all, what's more magical than a stunning rainbow stretching across the sky?


In [14]:
response = chat_engine.chat("What time is it?")
print(response.response)

We've already established that it's 3:00 AM in Singapore. But don't worry, you'll be waking up soon and starting your day! How about we chat about something to get you excited for tomorrow's presentation instead?
